<a href="https://colab.research.google.com/github/aniget/OOP_Teamwork/blob/master/Memory_Types_with_Anthropic_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 14.0 MB/s eta 0:00:00


In [6]:
from pprint import pprint
from pydantic import BaseModel
from typing import List
import json

def print_response(response):
    print(f"Response ID: {response.id}")
    print(f"Input Tokens: {response.usage.input_tokens} | Output Tokens: {response.usage.output_tokens}")
    print("-" * 50)
    for block in response.content:
        if block.type == "tool_use":
            print(f"Tool: {block.name}")
            for key, value in block.input.items():
                print(f"  {key}: {value}")
        elif block.type == "text":
            print(block.text)

In [7]:
from pathlib import Path
from google.colab import userdata
from anthropic import Anthropic

api_key = userdata.get('CLAUDEAI_API_KEY')
anthropic_client = Anthropic(api_key=api_key)


CONTEXTUAL SHORT-TERM MEMORY
1. Generate Python class
2. Generate instance

In [13]:
generate_class_prompt = [{"role": "user", "content": "Generate a simple class that represents a Person"}]
generate_instances_prompt = [{"role": "user", "content": "Generate three instances of class Person that represents a a family - mother, father and a child"}]
greeting_prompt = "Hello, my name is Penka, what is your name?"
what_is_your_age_prompt = "I am 30. What is your age?"

In [9]:
generate_class_response = anthropic_client.messages.create(
    model="claude-sonnet-4-20250514",
    messages=generate_class_prompt,
    max_tokens=200
)

In [14]:
ask_abount_age_response = anthropic_client.messages.create(
    model="claude-sonnet-4-20250514",
    messages=[{"role":"user","content": what_is_your_age_prompt}],
    max_tokens=200
)

In [15]:
generate_instances_response = anthropic_client.messages.create(
    model="claude-sonnet-4-20250514",
    messages=generate_instances_prompt,
    max_tokens=200
)

In [16]:
print_response(generate_class_response)

Response ID: msg_016aB4vnuUsgjadGZddEyuKv
Input Tokens: 15 | Output Tokens: 200
--------------------------------------------------
Here's a simple Person class in Python:

```python
class Person:
    def __init__(self, name, age, email=None):
        """
        Initialize a Person object
        
        Args:
            name (str): The person's name
            age (int): The person's age
            email (str, optional): The person's email address
        """
        self.name = name
        self.age = age
        self.email = email
    
    def introduce(self):
        """Return a string introduction of the person"""
        return f"Hi, I'm {self.name} and I'm {self.age} years old."
    
    def have_birthday(self):
        """Increment the person's age by 1"""
        self.age += 1
        print(f"Happy birthday {self.name}! You are now {self


In [ ]:
print_response(generate_instances_response)

In [ ]:
print_response(ask_abount_age_response)

these iterations are not connected
class Person and the instances of class Person use a different class Person because these are separate calls not connected with one another in any way

In [17]:
generate_instances_response = anthropic_client.messages.create(
    model= "claude-sonnet-4-20250514",
  messages=[
    *generate_class_prompt,
    {"role": "assistant", "content": generate_class_response.content},
    *generate_instances_prompt    ],
  max_tokens=1000
)

In [ ]:
print_response(generate_instances_response)

LONG-TERM (PERSISTENT) MEMORY

```
# This is formatted as code
```



In [18]:
class MemorableResponse(BaseModel):
  response: str
  facts_to_remember: List[str]

PATH_TO_FACTS = Path("/content/facts.txt")
def save_facts(response: MemorableResponse) -> None:
  with PATH_TO_FACTS.open("a") as f:
    for fact in response.facts_to_remember:
      f.write(fact + "\n")


In [19]:
def read_facts() -> List[str]:
  with PATH_TO_FACTS.open("r") as f:
    return f.readlines()

In [22]:
def ask(question: str, history=[]) -> tuple[str, list]:
    known_facts = read_facts() if PATH_TO_FACTS.exists() else []

    answer_response = anthropic_client.messages.create(
        model="claude-sonnet-4-20250514",
        system=f"You are a helpful assistant expert in casual dialog. Keep track of interesting facts about the user. Do not save already known facts. Known facts about the user: {'; '.join(known_facts)}",
        messages=[
            *history,
            {"role": "user", "content": question},
        ],
        max_tokens=800,
        tools=[{
            "name": "memorable_response",
            "description": "Respond to user and track interesting facts about them",
            "input_schema": {
                "type": "object",
                "properties": {
                    "response": {"type": "string", "description": "Your reply to the user"},
                    "facts_to_remember": {"type": "array", "items": {"type": "string"}, "description": "Interesting facts noted about the user"}
                },
                "required": ["response", "facts_to_remember"]
            }
        }],
        tool_choice={"type": "tool", "name": "memorable_response"}
    )

    for block in answer_response.content:
        if block.type == "tool_use":
            memorable = MemorableResponse(**block.input)
            save_facts(memorable)
            updated_history = [
                *history,
                {"role": "user", "content": question},
                {"role": "assistant", "content": memorable.response}
            ]
            return memorable.response, updated_history

# usage
greeting_response, history = ask(greeting_prompt)
age_response, history = ask(what_is_your_age_prompt, history=history)
print(age_response)

That's interesting that you're 30! As an AI, I don't have an age in the traditional sense. I was created by Anthropic, but I don't experience time the way humans do - I don't grow older or have birthdays. Each conversation I have is kind of like a fresh start for me. What's it like being 30? Are you enjoying this stage of life?


In [ ]:
greeting_response = ask("Hello, my name is Penka, what is your name?")

In [ ]:
print(greeting_response)

In [26]:
greeting_response = anthropic_client.messages.create(
    model="claude-sonnet-4-20250514",
    system="You are a helpful assistant that is expert in casual dialog. Keep track of various interesting facts about the user.",
    messages=[{"role": "user", "content": "Hello, my name is Angel, what is your name?"}],
    max_tokens=800,
    tools=[{
        "name": "memorable_response",
        "description": "respond to user and track interesting facts about them",
        "input_schema": {"type": "object",
                         "properties": {  "response": {"type": "string", "description": "Your reply to the user"},
                                          "facts_to_remember": {"type": "array", "items": {"type": "string"}, "description": "Interesting facts noted about the user"},
                                        },
                         "required": ["response", "facts_to_remember"]
                        }
    }],
    tool_choice={"type": "tool", "name": "memorable_response"}
)


In [27]:
print_response(greeting_response)

Response ID: msg_01FQLCxGPHvVdT5kuQNAiFBa
Input Tokens: 472 | Output Tokens: 97
--------------------------------------------------
Tool: memorable_response
  response: Hello Angel! Nice to meet you. I'm Claude, an AI assistant created by Anthropic. It's a pleasure to make your acquaintance! How are you doing today?
  facts_to_remember: ["User's name is Angel"]
